# Understanding `circuit_44` with `scheme='CCDQML'`

This notebook traces the call used by `Run.run_DQML('CCDQML', 3, X, 1, 100)` down to the actual quantum circuit:

```python
circuit = lambda weights, data_in: circuit_44('CCDQML', 3, weights, data_in)
```

The goal is not to train the full model again. Instead, we inspect one data sample, one weight vector, the parameter layout, the circuit drawing, the raw probability output, and the final class prediction rule.

In [1]:
from pathlib import Path
import sys
import pickle

import pennylane as qml
from pennylane import numpy as np

# Make this notebook work whether it is run from the repository root or from notebooks/.
ROOT = Path.cwd()
if not (ROOT / "DQML" / "DQML").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "DQML"))

from DQML.Circuit import circuit_44
from DQML.Circuitblocks import RandParam

print(f"Repository root: {ROOT}")
print("Imported circuit_44 successfully.")

Repository root: /Users/macos/Data/quantum/reproduce-distributed-quantum-ML-via-CC
Imported circuit_44 successfully.


## 1. Use the same high-level parameters as `Demo.ipynb`

The demo call is:

```python
Run.run_DQML('CCDQML', 3, X, 1, 100)
```

For understanding `circuit_44`, the important values are `scheme='CCDQML'`, `depth=3`, and one 8-feature data sample from `X`.

In [2]:
scheme = "CCDQML"
depth = 3

dataset_path = ROOT / "DQML" / "Dataset" / "Dataset1"
with dataset_path.open("rb") as f:
    X = pickle.load(f)

feats_train, y_train, feats_val, y_val, num_train = X
sample_index = 0
data_in = feats_train[sample_index]
label = y_train[sample_index]

print("Dataset pieces:")
print(f"  feats_train: {feats_train.shape}")
print(f"  y_train:     {y_train.shape}")
print(f"  feats_val:   {feats_val.shape}")
print(f"  y_val:       {y_val.shape}")
print(f"  num_train:   {num_train}")
print()
print(f"Using training sample #{sample_index}")
print(f"  data_in shape: {data_in.shape}")
print(f"  data_in: {np.round(data_in, 4)}")
print(f"  label: {label}")

Dataset pieces:
  feats_train: (1536, 8)
  y_train:     (1536,)
  feats_val:   (512, 8)
  y_val:       (512,)
  num_train:   1536

Using training sample #0
  data_in shape: (8,)
  data_in: [ 0.8052  0.6792 -0.7396  0.5543  1.1477 -1.0751  0.7825 -0.535 ]
  label: 1


## 2. What the lambda wrapper means

In `Machine.py`, the training code creates:

```python
circuit = lambda x, y: circuit_44(scheme, depth, x, y)
```

For this case, `x` means `weights` and `y` means `data_in`. The lambda freezes `scheme` and `depth`, so the optimizer only has to pass the changing values: weights and data.

In [3]:
circuit = lambda weights, data_in: circuit_44(scheme, depth, weights, data_in)

print("The wrapper above is equivalent to:")
print("def circuit(weights, data_in):")
print("    return circuit_44('CCDQML', 3, weights, data_in)")
print()
print("So the concrete call will be:")
print("circuit_44('CCDQML', 3, weights, one_8_feature_sample)")

The wrapper above is equivalent to:
def circuit(weights, data_in):
    return circuit_44('CCDQML', 3, weights, data_in)

So the concrete call will be:
circuit_44('CCDQML', 3, weights, one_8_feature_sample)


## 3. Create and inspect the trainable weights

`Machine.DQML` initializes `CCDQML` weights with:

```python
weights = RandParam((depth + 2) * 12)
```

At `depth=3`, this gives `(3 + 2) * 12 = 60` trainable parameters.

In [4]:
np.random.seed(7)
weights = RandParam((depth + 2) * 12)

print(f"depth: {depth}")
print(f"number of weights: {len(weights)}")
print(f"first 10 weights: {np.round(weights[:10], 4)}")

depth: 3
number of weights: 60
first 10 weights: [0.4795 4.9004 2.7546 4.5457 6.1449 3.3835 3.1486 0.4527 1.6867 3.1409]


## 4. How `circuit_44('CCDQML', depth, weights, data_in)` is structured

Inside `circuit_44`, the 8 input features are split into two 4-qubit groups:

```python
Ising(data_in[:4], [0,1,2,3], 1)
Ising(data_in[4:], [4,5,6,7], 1)
```

Then the `CCDQML` branch applies two classically-communicating convolution/pooling blocks:

```python
Convolution_Pooling_CC(weights[:40], [0,1,2,3], [4,5,6,7], depth)
Convolution_Pooling_CC(weights[40:60], [1,3], [5,7], depth)
```

Finally, it measures probability outcomes on wires `[3, 7]`.

In [5]:
first_block = weights[: 2 * (depth + 2) * 4]
second_block = weights[(depth + 2) * 8 : (depth + 2) * 12]

print("Input feature split:")
print(f"  data_in[:4] -> wires [0,1,2,3]: {np.round(data_in[:4], 4)}")
print(f"  data_in[4:] -> wires [4,5,6,7]: {np.round(data_in[4:], 4)}")
print()
print("Weight split for CCDQML depth=3:")
print(f"  first block  weights[:40]   -> shape {first_block.shape}, wires [0,1,2,3] <-> [4,5,6,7]")
print(f"  second block weights[40:60] -> shape {second_block.shape}, wires [1,3] <-> [5,7]")
print()
print("Inside each Convolution_Pooling_CC block:")
print("  1. apply Convolution separately to the two wire groups")
print("  2. measure one qubit")
print("  3. conditionally rotate a target qubit in the other group")
print("This is where the 'CC' classical communication appears.")

Input feature split:
  data_in[:4] -> wires [0,1,2,3]: [ 0.8052  0.6792 -0.7396  0.5543]
  data_in[4:] -> wires [4,5,6,7]: [ 1.1477 -1.0751  0.7825 -0.535 ]

Weight split for CCDQML depth=3:
  first block  weights[:40]   -> shape (40,), wires [0,1,2,3] <-> [4,5,6,7]
  second block weights[40:60] -> shape (20,), wires [1,3] <-> [5,7]

Inside each Convolution_Pooling_CC block:
  1. apply Convolution separately to the two wire groups
  2. measure one qubit
  3. conditionally rotate a target qubit in the other group
This is where the 'CC' classical communication appears.


## 5. Draw the circuit

This is the same style of drawing printed during training, but for one fixed sample and one fixed random weight vector.

In [6]:
print(qml.draw(circuit, decimals=2)(weights, data_in))

## 6. Run the circuit and inspect the raw output

`circuit_44` returns:

```python
qml.probs(wires=[3,7])
```

Because two wires are measured, the output has four probabilities: `[P(00), P(01), P(10), P(11)]`.

In [7]:
outcome = circuit(weights, data_in)

print("Raw circuit output probabilities:")
for state, prob in zip(["00", "01", "10", "11"], outcome):
    print(f"  P({state}) = {float(prob):.6f}")
print(f"  sum    = {float(np.sum(outcome)):.6f}")

Raw circuit output probabilities:
  P(00) = 0.252253
  P(01) = 0.224729
  P(10) = 0.284395
  P(11) = 0.238623
  sum    = 1.000000


## 7. Convert probabilities into a class prediction

For `CCDQML`, training uses a learned 4-value bias vector. At initialization, it starts as:

```python
bias = np.array([1.0, -1.0, -1.0, 1.0])
```

The model score is a weighted sum of the four output probabilities. The predicted label is the sign of that score.

In [8]:
initial_bias = np.array([1.0, -1.0, -1.0, 1.0])
score = np.dot(initial_bias, outcome)
prediction = np.sign(score)

print(f"initial bias: {initial_bias}")
print(f"score = bias dot probabilities = {float(score):.6f}")
print(f"prediction = sign(score) = {int(prediction)}")
print(f"true label for sample #{sample_index} = {int(label)}")

initial bias: [ 1. -1. -1.  1.]
score = bias dot probabilities = -0.018249
prediction = sign(score) = -1
true label for sample #0 = 1


## 8. Compare with the trained result saved by `Demo.ipynb`

If `DQML/CCDQML_depth3_1` exists, it contains the output of the first demo cell:

```python
[cost_list, acc_val_list, trained_weights, trained_bias]
```

The next cell uses those learned values on the same sample.

In [9]:
trained_path = ROOT / "DQML" / "CCDQML_depth3_1"

if trained_path.exists():
    with trained_path.open("rb") as f:
        cost_list, acc_list, trained_weights, trained_bias = pickle.load(f)

    trained_outcome = circuit(trained_weights, data_in)
    trained_score = np.dot(trained_bias, trained_outcome)
    trained_prediction = np.sign(trained_score)

    print("Loaded trained artifact:", trained_path)
    print(f"  training iterations stored: {len(cost_list)}")
    print(f"  final loss: {float(cost_list[-1]):.6f}")
    print(f"  final validation accuracy: {float(acc_list[-1]):.6f}")
    print(f"  trained_weights shape: {trained_weights.shape}")
    print(f"  trained_bias: {np.round(trained_bias, 6)}")
    print()
    print("Same sample with trained weights/bias:")
    for state, prob in zip(["00", "01", "10", "11"], trained_outcome):
        print(f"  P({state}) = {float(prob):.6f}")
    print(f"  trained score: {float(trained_score):.6f}")
    print(f"  trained prediction: {int(trained_prediction)}")
    print(f"  true label: {int(label)}")
else:
    print("No trained artifact found yet. Run the first cell in DQML/Demo.ipynb to create it.")

Loaded trained artifact: /Users/macos/Data/quantum/reproduce-distributed-quantum-ML-via-CC/DQML/CCDQML_depth3_1
  training iterations stored: 100
  final loss: 0.847886
  final validation accuracy: 0.742188
  trained_weights shape: (60,)
  trained_bias: [ 1.616713 -1.484121 -1.492739  1.44492 ]

Same sample with trained weights/bias:
  P(00) = 0.388685
  P(01) = 0.251396
  P(10) = 0.178277
  P(11) = 0.181641
  trained score: 0.251626
  trained prediction: 1
  true label: 1


## Mental model

For `CCDQML`, `circuit_44` does this:

1. Encode an 8-dimensional data sample into 8 qubits using two 4-qubit Ising embeddings.
2. Apply trainable convolution layers separately inside two quantum blocks.
3. Use mid-circuit measurement plus conditional rotations to communicate classical measurement information between the two blocks.
4. Compress from 8 wires to the final measured wires `[3, 7]`.
5. Return four probabilities `[P(00), P(01), P(10), P(11)]`.
6. Convert those probabilities into a class prediction with `sign(bias dot probabilities)`.